### Single-Trial Prediction Demo

- **File:** `single_trial_prediciton.ipynb`
- **Data:** Uses sample data for 3 participants in `/example_data/`..
- **Expected Output:** Model framework verification and prediction accuracy scores.
- **Expected Run Time:** Based on the manuscript settings (100 repetitions of 5-fold cross-validation), the expected run time is approximately 1–2 hours.

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_predict, KFold 
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.inspection import permutation_importance
import os
import numpy as np
import pandas as pd

def train_svm_classification_balanced_best_parameters_and_all_data_kernel_choose(
    X, y, 
    svm_kernel,
    test_size=0.2, 
    cv=5, 
    random_state=42, 
    param_grid=None,
    scoring='f1',
    permutation_test=False    
):
    """
    使用 SVM + class_weight='balanced' 进行二分类，缓解类别不平衡。
    在超参数搜索时，也可进一步搜索 class_weight=['balanced', None] 等等。
    
    参数:
      X, y: 特征矩阵和目标 (0/1 或其他二分类标签)
      test_size: 测试集比例
      cv: 交叉验证折数
      random_state: 随机种子
      param_grid: 超参网格 (SVC参数)；若为空则给一个示例
      scoring: 使用什么指标做模型选择 (如 'f1', 'balanced_accuracy', 'roc_auc' 等)
      permutation_test: 是否对训练标签进行置换检验
      
    返回:
      包含模型及预测结果等信息的字典，其中增加了最优参数、测试集预测结果，
      以及利用交叉验证对所有样本的预测结果。
    """

    # print(f"[随机种子]: {random_state}")

    if param_grid is None:
        if svm_kernel == 'rbf':
            param_grid = {
                'svc__kernel': ['rbf'],
                'svc__C': [0.1, 1, 10, 50, 100],
            }
        elif svm_kernel == 'linear':
            param_grid = {
                'svc__kernel': ['linear'],
                'svc__C': [0.1, 1, 10, 50, 100],
            }

    # 拆分数据集 (可使用 stratify 保持类分布)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, 
        test_size=test_size, 
        shuffle=True, 
        stratify=y,
        random_state=random_state
    )

    if permutation_test:
        y_train = np.random.permutation(y_train)

    # 构建管线: 标准化 + SVC(class_weight='balanced')
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('svc', SVC(random_state=random_state, probability=True, class_weight='balanced'))
    ])

    # 网格搜索
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring='recall',
        n_jobs=1  # 每个job使用1个CPU
    )
    grid_search.fit(X_train, y_train)
    
    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_  # 获取最优参数

    # 测试集评估（仅针对 train_test_split 得到的测试集）
    y_test_pred = best_model.predict(X_test)
    test_acc = accuracy_score(y_test, y_test_pred)
    report = classification_report(y_test, y_test_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_test, y_test_pred)

    # 输出预测正类概率（如果要绘制 ROC/PR 曲线）
    y_scores = best_model.predict_proba(X_test)[:, 1]

    # 特征重要性
    feature_importance = None
    if best_model.named_steps['svc'].kernel == 'linear':
        # 对于线性核，使用系数作为特征重要性
        feature_importance = best_model.named_steps['svc'].coef_[0]
    else:
        # 对于非线性核，使用排列特征重要性
        result = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=random_state)
        feature_importance = result.importances_mean

    # 使用 cross_val_predict 对所有数据进行交叉验证预测
    kf = KFold(n_splits=cv, shuffle=True, random_state=random_state)
    all_data_pred = cross_val_predict(best_model, X, y, cv=kf, n_jobs=1)
    # 如果需要预测概率，可以这样做（返回预测正类概率）：
    # all_data_pred_proba = cross_val_predict(best_model, X, y, cv=kf, n_jobs=1, method='predict_proba')[:, 1]

    # # 输出最优参数和预测结果，便于查看
    # print("最优参数:", best_params)
    # print("测试集预测结果:", y_test_pred)
    # print("所有数据的交叉验证预测结果:", all_data_pred)

    return {
        'test_accuracy': test_acc,
        'classification_report': report,
        'confusion_matrix': conf_matrix,
        'y_scores': y_scores,
        'feature_importance': feature_importance,
        'best_params': best_params,           # 返回最优参数
        'y_test_pred': y_test_pred,           # 返回测试集预测结果
        'y_test': y_test,
        'all_data_cv_prediction': all_data_pred  # 返回所有数据的交叉验证预测结果
    }

In [ ]:
import os
import numpy as np
import pandas as pd

import pickle
from joblib import Parallel, delayed
from itertools import combinations

if __name__ == '__main__':
    # 通过 single-trial RPE 的 beta-map 预测下一试次的 switch 行为，本代码使用的预测变量以人类组水平激活区域为 mask，在  single-trial RPE 的 beta-map 中提取激活区域的 beta 值作为特征 (激活体素 在 FDR-cluster 后按照 20 进行了筛选，剔除了一些激活区域较小的团块)。

    # 一些可以修改的参数
    # 1、LSS_file_name，确认特征的信息
    # 2、scoring，评估指标
    # 3、func_name，函数名 -- 便于保存的结果知道调用了哪些函数
    # 4、repeat_times，重复运行的次数
    # 5、permutation_test，是否进行置换检验  -- True/False True代表进行置换检验，查看的是特征的预测能力，False代表不进行置换检验，查看的是模型的 null 分布
    # 6、pkl_save_name，保存的文件名（根据设置的参数自动生成，无需更改）

    repeat_times = 100
    seeds_start = 1
    seeds_end = seeds_start + repeat_times
    permutation_test = False

    # '0404_LSS_pred_next_optimal_all_trials_monkey_con1_all_features',
    filename = '0613_use_human_regions_pred_best_ROI_PCA'
    
    kernel = 'rbf' 

    LSS_merge_data_path = os.path.join('/home/sangtian/Sangtian_Research/Data_Results/Proj_Awake_monkey/Data/Fig5_RF_pred_part', filename)
    LSS_file_name = [f for f in os.listdir(LSS_merge_data_path) if f.startswith('PCA_')]

    func_name = 'Random_effect_train_svm_balanced'
    pkl_save_name = f'{func_name}_{kernel}_perm_{permutation_test}_seed_start_{seeds_start}_end{seeds_end}.pkl'
    feature_names_save_path = os.path.join(LSS_merge_data_path, 'Random_effect_check_feature_names.txt')

    target_col = "switch"
    columns_to_drop=['subj_id', 'S', 'A', 'R', 'test', 'is_switch_point', 'optimal_choice']

    # 开始运行
    df = pd.read_csv(os.path.join(LSS_merge_data_path, LSS_file_name[0]))
    df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

    # X = df.drop(columns=target_col).values
    X_pre = df.drop(columns=target_col).values
    x_min, x_max = X_pre.min(), X_pre.max()
    df['PC_Random'] = np.random.uniform(low=x_min, high=x_max, size=len(df))
    X = df.drop(columns=target_col).values
    y = df[target_col].values

    # 获取特征名称
    feature_names = df.drop(columns=target_col).columns.tolist()
    # 保存特征名称
    with open(feature_names_save_path, 'w') as f:
        for feature in feature_names:
            f.write(f"{feature}\n")

    selected_features = ['PC_Random']

    # 累加每个特征
    # for feature in feature_names:
    for i, feature in enumerate(feature_names):
        print(i)
        selected_features.append(feature)  # 新加一列
        if i == 0:
            used_features = 'PC1'
        else:
            used_features = f"PC1_to_PC{i+1}"

        print('used feature name: ', used_features)

        # 构建当前数据
        # X_selected_df = df[selected_features].copy()
        # X_selected_df['PC_Random'] = np.random.uniform(low=x_min, high=x_max, size=len(df))

        X_selected = df[selected_features].values  # 提取当前特征的数据

        # 打印当前使用的特征名
        print(f"当前使用的特征数量: {len(selected_features)}")
        print(f"使用的特征列表: {selected_features}")
        print(f"X shape: {X_selected.shape}")

        # 保存结果的文件名
        pkl_save_name = f'{func_name}_{kernel}_perm_{permutation_test}_seed_start_{seeds_start}_end{seeds_end}_use_{used_features}.pkl'
        pkl_save_path = os.path.join(LSS_merge_data_path, pkl_save_name)

        # 并行
        results = Parallel(n_jobs=seeds_end-1)(delayed(train_svm_classification_balanced_best_parameters_and_all_data_kernel_choose)(
            X_selected, y,
            svm_kernel=kernel,
            test_size=0.2,
            cv=5,
            random_state=i,  # 从1到100迭代
            permutation_test=permutation_test
        ) for i in range(seeds_start, seeds_end))

        with open(pkl_save_path, 'wb') as f:
            pickle.dump(results, f)

        print("Results saved to svm_results.pkl")


In [ ]:
import os
import numpy as np
import pandas as pd

import pickle
from joblib import Parallel, delayed
from itertools import combinations


from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_predict, KFold 
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.inspection import permutation_importance
import os
import numpy as np
import pandas as pd

def train_svm_classification_balanced_best_parameters_and_all_data_kernel_choose(
    X, y, 
    svm_kernel,
    test_size=0.2, 
    cv=5, 
    random_state=42, 
    param_grid=None,
    scoring='f1',
    permutation_test=False    
):
    """
    Use SVM + class_weight='balanced' for binary classification to mitigate class imbalance.
    During hyperparameter search, class_weight=['balanced', None] etc. can also be further searched.
    
    Parameters:
      X, y: Feature matrix and target (0/1 or other binary classification labels)
      test_size: Test set proportion
      cv: Number of cross-validation folds
      random_state: Random seed
      param_grid: Hyperparameter grid (SVC parameters); if None, an example is provided
      scoring: Metric used for model selection (e.g., 'f1', 'balanced_accuracy', 'roc_auc', etc.)
      permutation_test: Whether to perform permutation test on training labels
      
    Returns:
      A dictionary containing model and prediction results, adding the best parameters, test set prediction results,
      and the prediction results for all samples using cross-validation.
    """

    # print(f"[Random seed]: {random_state}")

    if param_grid is None:
        if svm_kernel == 'rbf':
            param_grid = {
                'svc__kernel': ['rbf'],
                'svc__C': [0.1, 1, 10, 50, 100],
            }
        elif svm_kernel == 'linear':
            param_grid = {
                'svc__kernel': ['linear'],
                'svc__C': [0.1, 1, 10, 50, 100],
            }

    # Split dataset (can use stratify to maintain class distribution)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, 
        test_size=test_size, 
        shuffle=True, 
        stratify=y,
        random_state=random_state
    )

    if permutation_test:
        y_train = np.random.permutation(y_train)

    # Build pipeline: Standardization + SVC(class_weight='balanced')
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('svc', SVC(random_state=random_state, probability=True, class_weight='balanced'))
    ])

    # Grid search
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring='recall',
        n_jobs=1  # Use 1 CPU per job
    )
    grid_search.fit(X_train, y_train)
    
    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_  # Get the best parameters

    # Test set evaluation (only for the test set obtained from train_test_split)
    y_test_pred = best_model.predict(X_test)
    test_acc = accuracy_score(y_test, y_test_pred)
    report = classification_report(y_test, y_test_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_test, y_test_pred)

    # Output the predicted probability of the positive class (if ROC/PR curves are to be drawn)
    y_scores = best_model.predict_proba(X_test)[:, 1]

    # Feature importance
    feature_importance = None
    if best_model.named_steps['svc'].kernel == 'linear':
        # For linear kernel, use coefficients as feature importance
        feature_importance = best_model.named_steps['svc'].coef_[0]
    else:
        # For non-linear kernel, use permutation feature importance
        result = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=random_state)
        feature_importance = result.importances_mean

    # Use cross_val_predict to perform cross-validation prediction on all data
    kf = KFold(n_splits=cv, shuffle=True, random_state=random_state)
    all_data_pred = cross_val_predict(best_model, X, y, cv=kf, n_jobs=1)
    # If probability predictions are needed, do this (returns predicted positive class probabilities):
    # all_data_pred_proba = cross_val_predict(best_model, X, y, cv=kf, n_jobs=1, method='predict_proba')[:, 1]

    # # Output best parameters and prediction results for easy viewing
    # print("Best parameters:", best_params)
    # print("Test set prediction results:", y_test_pred)
    # print("Cross-validation prediction results for all data:", all_data_pred)

    return {
        'test_accuracy': test_acc,
        'classification_report': report,
        'confusion_matrix': conf_matrix,
        'y_scores': y_scores,
        'feature_importance': feature_importance,
        'best_params': best_params,           # Return the best parameters
        'y_test_pred': y_test_pred,           # Return the test set prediction results
        'y_test': y_test,
        'all_data_cv_prediction': all_data_pred  # Return the cross-validation prediction results for all data
    }


if __name__ == '__main__':
    # Predict switch behavior of the next trial through the single-trial RPE beta-map. The predictor variables used in this code are masked by the human group-level activation regions, and the beta values of the activation regions are extracted as features from the single-trial RPE beta-map (activated voxels were filtered by 20 after FDR-cluster to remove some small activation clusters).

    # Some modifiable parameters
    # 1. LSS_file_name, confirm feature information
    # 2. scoring, evaluation metric
    # 3. func_name, function name -- helps to know which functions were called from the